# Taller integrador · ¿De dónde sale el 27.6 %?

**Estadística Descriptiva e Inferencial** · Taller · 120 minutos

Reconstruir la cifra oficial de pobreza del Perú desde el microdato, y después
averiguar qué tan segura es.

---

## El titular

> «La pobreza monetaria en el Perú afectó al **27.6 %** de la población en 2024,
> frente al 29.0 % del año anterior.»

Ese número sale de la **ENAHO**, una encuesta a unos 34 000 hogares. Hoy vas a hacer
cuatro cosas con él:

1. **Reproducirlo** exactamente, desde el gasto de cada hogar.
2. **Ponerle un intervalo**, y descubrir que es más ancho de lo que parece.
3. **Comparar grupos**: urbano contra rural, y 2024 contra 2023.
4. **Escribir el informe** que un comité podría auditar.

## Lo que este taller integra

| Bloque | Min | Qué usas de las clases anteriores |
|---|---|---|
| 0 · El titular y los datos | 12 | — |
| 1 · **Factores de expansión** | 28 | tema nuevo |
| 2 · Reconstruir la cifra | 20 | Clases 1, 2 y 3 |
| 3 · ¿Qué tan seguro es? | 22 | Clases 3 y 4 |
| 4 · Comparar grupos | 25 | Clases 5 y 6 |
| 5 · El informe | 13 | Clases 4, 5 y 6 |

> **Formato:** este notebook se trabaja **en vivo**. Yo ejecuto y explico; tú sigues y
> completas los huecos marcados con `# ── TU CÓDIGO ──`. Las celdas de verificación te
> avisan si un número no coincide.

---
## Celda 0 · Preparación y carga de datos

Ejecuta esta celda. Intenta descargar el dataset del repo del curso; si falla, te
permite subirlo a mano.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (9, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})
pd.set_option("display.width", 140)

# ── URL del dataset ──────────────────────────────────────────────────────
URL = ("https://github.com/josefrodrim/Estad-stica-Descriptiva-E-Inferencial/"
       "blob/main/Taller_7/Data/enaho_taller.csv.gz")

def a_raw(url):
    """GitHub sirve HTML en /blob/. Lo convierte al enlace de descarga directa."""
    if "github.com" in url and "/blob/" in url:
        url = (url.replace("github.com", "raw.githubusercontent.com")
                  .replace("/blob/", "/"))
    return url

def cargar():
    try:
        d = pd.read_csv(a_raw(URL))
        print("Datos cargados desde el repo del curso.")
        return d
    except Exception as e:
        print(f"No se pudo descargar ({type(e).__name__}).")
        print("Sube el archivo enaho_taller.csv.gz con el botón de archivos de Colab,")
        print("o ejecuta: from google.colab import files; files.upload()")
        try:
            return pd.read_csv("enaho_taller.csv.gz")
        except Exception:
            raise SystemExit("Carga el archivo y vuelve a ejecutar esta celda.")

df = cargar()
print(f"\n{len(df):,} hogares · {len(df.columns)} variables · años {sorted(df['anio'].unique())}")

# ── Verificador ──────────────────────────────────────────────────────────
def check(nombre, obtenido, esperado, tol=1e-4):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada (None)")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}: obtenido = {float(obtenido):,.4f} | "
          f"esperado = {float(esperado):,.4f}")
    if not ok:
        print("      -> revisa este paso antes de continuar.")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"      -> {pista}")
    return bool(cond)

# ── Mediana ponderada: numpy no la trae ──────────────────────────────────
def mediana_ponderada(x, w):
    """Mediana de x ponderada por w, para estimar cuantiles poblacionales."""
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    o = np.argsort(x); x, w = x[o], w[o]
    return float(np.interp(0.5, np.cumsum(w) / w.sum(), x))

### El diccionario de este dataset

Es un extract de la **ENAHO 2023 y 2024**, módulo 34 (Sumarias) unido al módulo 01
(NBI, solo 2024). Un registro por hogar.

| Variable | Qué es |
|---|---|
| `anio` | 2023 o 2024 |
| `conglome` | **conglomerado**: el área geográfica donde se sortearon los hogares |
| `estrato` | estrato geográfico (1–5 urbano, 6–8 rural) |
| `dominio`, `dominio_nom` | 8 dominios: Costa/Sierra/Selva y Lima Metropolitana |
| `area` | urbano o rural |
| `dpto` | departamento |
| `mieperho` | número de miembros del hogar |
| `gashog2d` | **gasto total anual del hogar**, en soles |
| `linea` | **línea de pobreza** mensual per cápita, según dominio |
| `linpe` | línea de pobreza *extrema* (solo alimentaria) |
| `pobreza` | 1 = pobre extremo, 2 = pobre no extremo, 3 = no pobre |
| `pobre` | 1 si el hogar es pobre (`pobreza <= 2`) |
| `factor07` | **factor de expansión**: a cuántos hogares del país representa |
| `w_hog` | = `factor07` (peso de hogares) |
| `w_per` | = `factor07 × mieperho` (peso de **personas**) |
| `nbi1`…`nbi5`, `n_nbi`, `pobre_nbi` | necesidades básicas insatisfechas (solo 2024) |

**Fuente:** INEI, ENAHO, módulos `906-Modulo34`, `966-Modulo34` y `966-Modulo01`.

---
# Bloque 0 · Mira los datos antes de tocarlos  ·  12 min

Regla que arrastramos desde la Clase 1: primero se mira, después se calcula.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
d24 = df[df.anio == 2024].copy()
d23 = df[df.anio == 2023].copy()

print(f"2024: {len(d24):,} hogares    2023: {len(d23):,} hogares")
print()
print("Distribución de la muestra 2024 por dominio:")
t = (d24.groupby("dominio_nom")
        .agg(hogares=("pobre", "size"), pobres_muestra=("pobre", "mean"))
        .assign(pobres_muestra=lambda x: (100*x.pobres_muestra).round(1))
        .sort_values("hogares", ascending=False))
print(t.to_string())
print()
print("Un primer intento ingenuo de la tasa de pobreza:")
print(f"  hogares pobres / hogares totales = {100*d24['pobre'].mean():.2f} %")
print()
print("Pero el titular dice 27.6 %. Y ese numero es de PERSONAS, no de hogares,")
print("y esta EXPANDIDO a la poblacion. Esas dos cosas son el bloque 1.")

### La serie que hay que tener en la cabeza

Pobreza monetaria en el Perú, población afectada (INEI):

| Año | 2019 | 2020 | 2021 | 2022 | 2023 | 2024 |
|---|---|---|---|---|---|---|
| % | 20.2 | 30.1 | 25.9 | 27.5 | 29.0 | **27.6** |

El salto de 2020 es la pandemia. Lo que vamos a estudiar hoy es la última columna: si
esa caída de 29.0 a 27.6 es real o cabe dentro del ruido del muestreo.

*(Apéndice opcional al final del notebook: comparar Perú con el resto de Sudamérica
usando la API del Banco Mundial.)*

---
# Bloque 1 · Factores de expansión  ·  28 min  ·  **TEMA NUEVO**

Aquí está la pieza que faltaba en todo el curso.

## El problema

La ENAHO **no** es un muestreo aleatorio simple. Es estratificado y por conglomerados,
y sobremuestrea zonas rurales y departamentos pequeños a propósito: si sorteara al azar
puro, Madre de Dios saldría con 30 hogares y no se podría decir nada de ese departamento.

Consecuencia: **un hogar de la muestra no vale lo mismo que otro.** Un hogar rural de
Amazonas puede representar a 40 hogares del país; uno de Lima, a 1 500.

## La solución

`factor07` dice a cuántos hogares del país representa cada hogar de la muestra. Para
estimar cualquier cosa a nivel país hay que **ponderar** por ese factor.

$$\bar{x}_{ponderado} = \frac{\sum w_i x_i}{\sum w_i}$$

En numpy: `np.average(x, weights=w)`.

### Ejercicio 1.1 — ¿A cuánto expande la muestra?

Si `factor07` es «a cuántos hogares representa cada hogar», entonces su suma debería ser
**el total de hogares del Perú**. Compruébalo.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
total_hogares   = None   # suma de w_hog en 2024
total_personas  = None   # suma de w_per en 2024

print(f"hogares  estimados: {total_hogares}")
print(f"personas estimadas: {total_personas}")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("hogares estimados del Perú", total_hogares, 10_360_811, tol=1),
     check("personas estimadas del Perú", total_personas, 34_482_699, tol=2),
     check_bool("y el resultado es plausible: ~10.4 M de hogares y ~34.5 M de personas",
                9e6 < total_hogares < 12e6 and 32e6 < total_personas < 36e6)]
print()
print("Ese es el sentido de 'expandir': la muestra habla por todo el pais.")
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — Las tres tasas de pobreza

Ahora el punto central del bloque. Calcula la tasa de tres maneras:

1. **Sin ponderar:** hogares pobres / hogares de la muestra.
2. **Ponderada por hogares:** usando `w_hog`.
3. **Ponderada por personas:** usando `w_per`.

Solo una de las tres es la cifra oficial. Averigua cuál.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
tasa_sin_pond   = None   # en PORCENTAJE
tasa_pond_hog   = None
tasa_pond_per   = None

print(f"sin ponderar      : {tasa_sin_pond} %")
print(f"ponderada hogares : {tasa_pond_hog} %")
print(f"ponderada personas: {tasa_pond_per} %")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("tasa sin ponderar (%)", tasa_sin_pond, 20.2190, tol=1e-3),
     check("tasa ponderada por hogares (%)", tasa_pond_hog, 21.8509, tol=1e-3),
     check("tasa ponderada por personas (%) = LA OFICIAL", tasa_pond_per, 27.5795, tol=1e-3)]
print()
print("Lo que hay que llevarse del bloque:")
print("  ignorar los pesos habria dado 20.2 % en lugar de 27.6 %.")
print("  Son 7.4 puntos porcentuales, o 2.5 millones de personas.")
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2")

### Para discutir (2 min)

Un analista te entrega un reporte que dice «la pobreza es 20.2 %» y su código es
correcto: sumó bien, dividió bien, no hay bugs.

¿Qué le dirías? ¿Y qué tipo de error es este — de cálculo, de datos, o de diseño?

*(Es de diseño. Y es el mismo error de la slide del sesgo de selección de la Clase 4:
tratar una muestra no aleatoria como si lo fuera. Ningún test unitario lo detecta.)*

---
# Bloque 2 · Reconstruir la cifra desde cero  ·  20 min

Ya reprodujimos el 27.6 %, pero usando la variable `pobreza` que el INEI ya calculó.
Ahora la construimos nosotros, desde el gasto de cada hogar.

**La definición oficial, completa:**

> Un hogar es pobre si su **gasto per cápita mensual** está por debajo de la **línea de
> pobreza** de su dominio geográfico.

Dos cosas que revive esto: el gasto es una variable **continua y lognormal** (Clase 3), y
la comparación contra un umbral produce una variable Bernoulli (Clase 1).

> **Precisión importante:** la línea de pobreza **no es un cuantil**. Un cuantil se define
> por su posición dentro de una distribución (el percentil 27, por ejemplo). La línea de
> pobreza es un **umbral monetario absoluto**: el costo de una canasta básica de consumo
> que el INEI calcula para cada dominio. Que el 27.6 % de la población quede por debajo es
> el *resultado* de aplicar ese umbral, no su definición.

### Ejercicio 2.1 — Construye el gasto per cápita mensual

`gashog2d` es el gasto **anual** del **hogar**. Necesitas gasto **mensual** por
**persona**.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
d24["gpc"] = None    # gasto per cápita MENSUAL

print(d24[["gashog2d", "mieperho", "gpc", "linea"]].head())

### Ejercicio 2.2 — ¿Es lognormal?

Aplica el diagnóstico de la Clase 3: asimetría y exceso de curtosis, sobre la variable
y sobre su logaritmo.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
asim_gpc  = None   # asimetría del gasto per cápita
curt_gpc  = None   # exceso de curtosis
asim_log  = None   # asimetría de log(gpc)
curt_log  = None

print(f"gpc      : asim={asim_gpc} curt={curt_gpc}")
print(f"log(gpc) : asim={asim_log} curt={curt_log}")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("asimetría del gasto", asim_gpc, 3.8019, tol=1e-3),
     check("exceso de curtosis del gasto", curt_gpc, 41.0265, tol=1e-2),
     check_bool("el logaritmo queda casi simétrico (|asimetría| < 0.15)",
                abs(asim_log) < 0.15),
     check_bool("y con curtosis casi normal (|exceso| < 0.15)", abs(curt_log) < 0.15)]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")

### Ejercicio 2.3 — Construye la pobreza y compárala con la oficial

Ahora el momento del taller: define pobre como `gpc < linea` y compara tu variable con
la del INEI, hogar por hogar.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
d24["mi_pobre"] = None   # 1 si gpc < linea

coincidencia = None      # % de hogares donde tu variable coincide con 'pobre'
mi_tasa      = None      # tu tasa de pobreza de personas, en %

print(f"coincidencia: {coincidencia} %")
print(f"mi tasa     : {mi_tasa} %")

In [ ]:
# ── VERIFICACIÓN 2.3 ─────────────────────────────────────────────────────
r = [check("coincidencia con la variable oficial (%)", coincidencia, 100.0, tol=0.01),
     check("tu tasa de pobreza de personas (%)", mi_tasa, 27.5795, tol=1e-3)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.3")

---
# Bloque 3 · ¿Qué tan seguro es ese 27.6 %?  ·  22 min

El 27.6 % es una **estimación** hecha con 33 691 hogares de un país de 34 millones de
personas. Toca ponerle el intervalo de la Clase 4.

Y aquí va a pasar algo: el intervalo que aprendimos **no sirve tal cual**.

### Ejercicio 3.1 — El intervalo ingenuo

Empecemos con lo que sabemos: intervalo para una proporción, con la fórmula de la
Clase 4 (`p ± 1.96 · √(p(1−p)/n)`).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
p_hat = np.average(d24["pobre"], weights=d24["w_per"])   # la estimación puntual
n     = len(d24)

ee_ingenuo = None    # error estándar suponiendo muestreo aleatorio simple
ic_lo_ing  = None    # límite inferior en PORCENTAJE
ic_hi_ing  = None

print(f"IC ingenuo 95 %: [{ic_lo_ing}, {ic_hi_ing}] %")

### Ejercicio 3.2 — El problema de los conglomerados

Mira la estructura real de la muestra antes de seguir.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
g = d24.groupby("conglome")
print(f"conglomerados en 2024      : {d24['conglome'].nunique():,}")
print(f"hogares por conglomerado   : media {g.size().mean():.1f}  rango [{g.size().min()}, {g.size().max()}]")
print()
# ¿se parecen los hogares dentro de un mismo conglomerado?
tasa_cong = g["pobre"].mean()
print("Tasa de pobreza DENTRO de cada conglomerado:")
print(f"  conglomerados con 0 % de pobres  : {100*(tasa_cong==0).mean():.1f} %")
print(f"  conglomerados con 100 % de pobres: {100*(tasa_cong==1).mean():.1f} %")
print(f"  conglomerados 'mixtos'           : {100*((tasa_cong>0)&(tasa_cong<1)).mean():.1f} %")
print()
hom = 100*((tasa_cong==0).mean()+(tasa_cong==1).mean())
print(f"Casi el {hom:.0f} % de los conglomerados son HOMOGENEOS: o todos pobres o ninguno.")
print("Es una proporcion enorme, y significa que los hogares vecinos SE PARECEN.")
print("Por lo tanto cada hogar")
print("nuevo dentro del mismo conglomerado aporta MENOS informacion nueva de la que")
print("aportaria un hogar sorteado al azar en todo el pais.")
print()
print("La slide 17 de la Clase 4 lo anticipaba: 'con datos por conglomerados el EE real")
print("es mayor que s/raiz(n) y el intervalo sale falsamente angosto'.")

### Ejercicio 3.3 — El error estándar correcto, y el efecto de diseño

La fórmula que respeta el diseño (método del *conglomerado último*):

$$\widehat{Var}(\hat p) = \frac{m}{m-1} \cdot \frac{\sum_c a_c^2}{(\sum_i w_i)^2}
\qquad \text{con} \qquad a_c = \sum_{i \in c} w_i (y_i - \hat p)$$

Donde `m` es el número de conglomerados y `a_c` es la contribución de cada uno.

**La idea:** en lugar de tratar los 33 691 hogares como independientes, trata los 5 359
**conglomerados** como las unidades independientes.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
w = d24["w_per"]
total_w = w.sum()

# a_c: por cada conglomerado, la suma de w_i * (y_i - p_hat)
a_c = None            # una Serie con un valor por conglomerado
m   = None            # número de conglomerados
var_diseno = None
ee_diseno  = None

deff = None           # efecto de diseño = (ee_diseno / ee_ingenuo)²
n_efectivo = None     # n / deff

print(f"EE con diseño = {ee_diseno} | DEFF = {deff} | n efectivo = {n_efectivo}")

In [ ]:
# ── VERIFICACIÓN 3.3 ─────────────────────────────────────────────────────
r = [check("número de conglomerados", m, 5359, tol=0),
     check("EE ingenuo (pp)", 100*ee_ingenuo, 0.2435, tol=1e-2),
     check("EE con diseño (pp)", 100*ee_diseno, 0.5253, tol=1e-2),
     check("efecto de diseño (DEFF)", deff, 4.6551, tol=0.05),
     check("n efectivo", n_efectivo, 7238, tol=50),
     check_bool("el EE con diseño es mayor que el ingenuo", ee_diseno > ee_ingenuo)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.3")

### Y para la media del gasto

Lo mismo aplica a cualquier estimación, no solo a proporciones. Reportar el gasto medio
sin corregir por diseño es el mismo error.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
gm = np.average(d24["gpc"], weights=d24["w_per"])
ee_g_ing = d24["gpc"].std(ddof=1) / np.sqrt(n)
a_g = d24.groupby("conglome").apply(lambda s: (s["w_per"] * (s["gpc"] - gm)).sum())
ee_g_dis = np.sqrt((m/(m-1)) * (a_g**2).sum() / total_w**2)

print(f"Gasto per capita medio del Peru = S/ {gm:.2f} al mes")
print(f"  IC ingenuo : [S/ {gm-1.96*ee_g_ing:.2f}, S/ {gm+1.96*ee_g_ing:.2f}]")
print(f"  IC diseño  : [S/ {gm-1.96*ee_g_dis:.2f}, S/ {gm+1.96*ee_g_dis:.2f}]")
print(f"  DEFF del gasto medio = {(ee_g_dis/ee_g_ing)**2:.2f}")
print()
print("Nota de la Clase 4: en una poblacion tan asimetrica, la MEDIANA es un mejor")
print("resumen del bienestar tipico que la media. Pero hay que ponderarla igual que todo:")
print()
print(f"  mediana MUESTRAL  : S/ {d24['gpc'].median():.2f}   <- describe la MUESTRA")
print(f"  mediana PONDERADA : S/ {mediana_ponderada(d24['gpc'], d24['w_per']):.2f}   <- describe al PAIS")
print()
print("Son unos S/ 45 de diferencia, un 7 %. La mediana muestral esta sesgada al alza")
print("porque las zonas mas pobres estan sobremuestreadas y pesan de mas en la muestra cruda.")

---
# Bloque 4 · Comparar grupos  ·  25 min

Dos preguntas que un comité haría de inmediato:

1. **¿Es distinta la pobreza entre lo urbano y lo rural?**
2. **¿Bajó la pobreza de 2023 a 2024?**

La primera es fácil. La segunda es donde se juega el taller.

### Ejercicio 4.1 — Urbano contra rural

Primero identifica el diseño (Clase 6, slide 5) y después elige la prueba.

> **Aviso metodológico, y hay que decirlo en voz alta:** las **tasas** que reportamos abajo
> están ponderadas, pero la **prueba de hipótesis** (Welch) y el **tamaño del efecto** (d de
> Cohen) se calculan sobre la muestra **sin ponderar**. Son dos cosas distintas:
>
> - las tasas estiman al **país**;
> - el Welch compara los dos grupos **de la muestra**.
>
> Hacer pruebas de hipótesis con diseño complejo requiere métodos específicos que no
> cubrimos en el curso. Para clase la aproximación sirve —la conclusión no cambia—, pero
> **no la presentes como estimación oficial de encuesta compleja**.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
u = d24[d24.area == "urbano"]
r_ = d24[d24.area == "rural"]

pob_urbano = None    # tasa ponderada por personas, en %
pob_rural  = None

p_levene = None      # ¿varianzas iguales en el gasto?
res_welch = None     # t de Welch sobre gpc
d_cohen  = None      # tamaño del efecto

print(f"urbano {pob_urbano} % | rural {pob_rural} %")

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("pobreza urbana (%)", pob_urbano, 24.8070, tol=1e-2),
     check("pobreza rural (%)", pob_rural, 39.3028, tol=1e-2),
     check("d de Cohen del gasto", d_cohen, 0.7512, tol=1e-3),
     check_bool("Levene rechaza igualdad de varianzas", p_levene < 0.01),
     check_bool("y por eso usaste Welch", res_welch.df < len(u)+len(r_)-2)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — ¿Bajó la pobreza? (el ejercicio del taller)

El titular dice que la pobreza bajó de 29.0 % a 27.6 %: **1.5 puntos porcentuales**.

La pregunta es si esa caída es real o cabe dentro del ruido del muestreo. Y aquí importa
todo lo del bloque 3: si usas el error estándar ingenuo, vas a llegar a una conclusión
distinta que si usas el correcto.

Calcula **las dos versiones** y compáralas.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def tasa_y_ee(d):
    """Devuelve (p, ee_ingenuo, ee_diseno) de la tasa de pobreza de personas."""
    # TU CÓDIGO: reutiliza lo del bloque 3
    return None, None, None

p23, e23_ing, e23_dis = tasa_y_ee(d23)
p24, e24_ing, e24_dis = tasa_y_ee(d24)

dif = None            # p24 - p23
z_honesto = None      # usando los EE de diseño
z_ingenuo = None      # usando los EE ingenuos

print(f"diferencia = {dif} | z honesto = {z_honesto} | z ingenuo = {z_ingenuo}")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("pobreza 2023 (%)", 100*p23, 29.0459, tol=1e-2),
     check("pobreza 2024 (%)", 100*p24, 27.5795, tol=1e-2),
     check("caída (pp)", 100*dif, -1.4665, tol=1e-2),
     check("z con diseño", z_honesto, -1.9974, tol=1e-2),
     check_bool("con diseño la caída es significativa por poco (0.01 < p < 0.05)",
                0.01 < 2*stats.norm.sf(abs(z_honesto)) < 0.05),
     check_bool("y sin diseño parecería mucho más contundente",
                abs(z_ingenuo) > 2*abs(z_honesto))]
print()
print("4.2 OK" if all(r) else "Revisa 4.2")

### Un matiz de diseño que no conviene esconder

Traté las muestras de 2023 y 2024 como **independientes**, que es lo que hace el INEI en
sus publicaciones. Pero la ENAHO tiene un **panel rotativo**: parte de los hogares se
vuelve a visitar al año siguiente.

Compruébalo.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
for d in (d23, d24):
    d["id"] = (d["conglome"].astype(str) + "-" + d["vivienda"].astype(str)
               + "-" + d["hogar"].astype(str))
comunes = set(d23["id"]) & set(d24["id"])
print(f"identificadores presentes en los dos años: {len(comunes):,}")
print(f"  = {100*len(comunes)/len(d24):.1f} % de la muestra de 2024")
print()

# PASO 1: ¿son REALMENTE los mismos hogares, o coincidencias del marco muestral?
mg = d23.merge(d24, on="id", suffixes=("_23", "_24"))
print("¿Son de verdad los mismos hogares?")
print(f"  mismo dominio : {100*(mg.dominio_23==mg.dominio_24).mean():.1f} %")
print(f"  mismo estrato : {100*(mg.estrato_23==mg.estrato_24).mean():.1f} %")
print(f"  correlacion del numero de miembros entre anios: {mg['mieperho_23'].corr(mg['mieperho_24']):.3f}")
print("  -> Si fueran hogares distintos que coinciden por azar del marco, el dominio")
print("     tambien coincidiria, pero mieperho NO estaria correlacionado. Lo esta.")
print("     Confirmado: es panel rotativo real.")
print()

# PASO 2: medir la correlacion de la condicion de pobreza entre anios
r_panel = np.corrcoef(mg["pobre_23"], mg["pobre_24"])[0, 1]
frac = len(mg) / len(d24)
print(f"Correlacion de 'pobre' en los hogares del panel: r = {r_panel:.3f}")
print(f"Fraccion solapada: f = {frac:.3f}")
print()

# PASO 3: recalcular el EE de la diferencia INCORPORANDO la covarianza
cov_aprox = frac * r_panel * e23_dis * e24_dis
ee_corr = np.sqrt(e23_dis**2 + e24_dis**2 - 2*cov_aprox)
z_corr = dif / ee_corr
p_corr = 2 * stats.norm.sf(abs(z_corr))

print(f"{'':32}{'EE dif':>10}{'z':>9}{'p':>10}")
print("-" * 62)
print(f"{'asumiendo independencia':32}{100*ee_dif_dis:10.4f}{z_honesto:9.3f}{2*stats.norm.sf(abs(z_honesto)):10.4f}")
print(f"{'con la covarianza medida':32}{100*ee_corr:10.4f}{z_corr:9.3f}{p_corr:10.4f}")
print("-" * 62)
print()
if cov_aprox > 0:
    print("La covarianza resulto POSITIVA. Eso significa que asumir independencia")
    print("SOBREestima el error estandar, es decir, somos conservadores. La conclusion")
    print("del ejercicio anterior se sostiene... pero ahora esta COMPROBADA, no supuesta.")
else:
    print("La covarianza NO resulto positiva: el argumento de que somos conservadores")
    print("NO se sostiene con estos datos.")
print()
print("Y fijate en lo que importa de verdad: el p se movio de 0.046 a 0.034 por")
print("cambiar UN supuesto. Sigue siendo fronterizo. Eso es lo que significa que un")
print("resultado sea 'sensible al metodo', y es la razon para no titular")
print("'demostramos que la pobreza bajo'.")
print()
print("Leccion de la Clase 6, slide 5: identifica el diseno antes de elegir la prueba.")
print("Y a veces el mundo real no cae limpio en ninguna de las tres casillas.")

In [ ]:
# ── VERIFICACIÓN 4.2b ────────────────────────────────────────────────────
r = [check_bool("confirmaste que el solapamiento es panel real (correlación de mieperho > 0.5)",
                mg["mieperho_23"].corr(mg["mieperho_24"]) > 0.5),
     check_bool("la correlación de pobreza entre años es positiva", r_panel > 0),
     check_bool("y por tanto asumir independencia es conservador", cov_aprox > 0),
     check_bool("pero el resultado sigue siendo fronterizo (0.01 < p < 0.05)",
                0.01 < p_corr < 0.05,
                "si sale fuera de ese rango, la conclusión del bloque cambia")]
print()
print("Lo que este ejercicio enseña, y vale mas que el numero:")
print("  afirmar 'esto es conservador' SIN medirlo es exactamente el tipo de")
print("  razonamiento plausible-pero-no-verificado que este curso combate.")
print("  Medirlo cuesta cinco lineas.")
print()
print("4.2b OK" if all(r) else "Revisa 4.2b")

### Ejercicio 4.3 — Comparar los ocho dominios

Última pregunta: ¿hay diferencias entre los ocho dominios geográficos? Aquí vuelve el
problema de las comparaciones múltiples (Clase 5).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
tabla_dom = None   # DataFrame con dominio_nom, n, y tasa de pobreza ponderada

n_comparaciones = None   # cuántos pares hay con 8 dominios
alpha_bonf      = None   # 0.05 / n_comparaciones

print(tabla_dom)
print(f"comparaciones: {n_comparaciones} | alpha Bonferroni: {alpha_bonf}")

In [ ]:
# ── VERIFICACIÓN 4.3 ─────────────────────────────────────────────────────
r = [check("número de comparaciones por pares", n_comparaciones, 28, tol=0),
     check("alpha de Bonferroni", alpha_bonf, 0.05/28, tol=1e-6),
     check_bool("identificaste Sierra Norte como el dominio más pobre",
                tabla_dom.index[0] == "Sierra Norte"),
     check_bool("y Costa Centro como el menos pobre",
                tabla_dom.index[-1] == "Costa Centro")]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.3")

---
# Bloque 5 · El informe  ·  13 min

Todo lo anterior no sirve si el informe no lo comunica. Este bloque **no tiene respuesta
numérica única**: se te pide escribir.

Recuerda la slide de redacción de la Clase 6: diagnóstico de supuestos, prueba con sus
grados de libertad, efecto en unidades reales, tamaño del efecto, cuántas comparaciones,
y **qué no permite concluir la muestra**.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Escribe el informe de tres párrafos que entregarías a un comité.
# Debe incluir: la cifra, su intervalo CON diseño, la comparación entre años
# con su lectura honesta, y al menos una limitación.

informe = """
(escribe aquí tu informe)
"""

print(informe)

In [ ]:
# ── VERIFICACIÓN 5 ──────────────────────────────────────────────────────
r = [check_bool("escribiste un informe sustantivo", len(informe.strip()) > 300),
     check_bool("mencionas un intervalo de confianza",
                any(k in informe.upper() for k in ["IC ", "INTERVALO", "IC:"])),
     check_bool("mencionas el efecto de diseño o el n efectivo",
                any(k in informe.upper() for k in ["DEFF", "DISENO", "DISEÑO", "EFECTIVO"])),
     check_bool("incluyes al menos una limitación",
                any(k in informe.upper() for k in ["LIMITACI", "NO PERMITE", "PRUDENTE", "NO CAPTURA"]))]
print()
print("TALLER COMPLETO" if all(r) else "Completa el informe del bloque 5")

---
# Cierre

### Checklist de salida

- [ ] Sé qué es un factor de expansión y por qué sin él la cifra está mal.
- [ ] Sé que la pobreza oficial se pondera por **personas**, no por hogares.
- [ ] Reconstruí la cifra oficial del país desde el gasto de cada hogar.
- [ ] Sé calcular un efecto de diseño y explicar qué es el n efectivo.
- [ ] Sé que un IC ingenuo sobre datos por conglomerados es falsamente angosto.
- [ ] Identifiqué el diseño antes de elegir la prueba, y noté que era mixto.
- [ ] Escribí un informe que dice también lo que no se puede concluir.

### Los cinco números del taller

| | |
|---|---|
| Cifra oficial reproducida | **27.58 %** (INEI publica 27.6 %) |
| Coincidencia de mi variable con la oficial | **100.00 %** |
| Sin ponderar habría dicho | 20.22 % → error de **7.4 pp** |
| Efecto de diseño | **DEFF = 5.76** → n efectivo **5 845** de 33 691 |
| ¿Bajó la pobreza? | p = **0.046** con diseño, p = **0.0000027** sin él |

### Lo que este taller demostró

Las seis clases anteriores no fueron ejercicios de pizarra. Cada una apareció aquí, con
datos del INEI, y en cada punto la decisión metodológica **cambió la respuesta**:

- ignorar los pesos habría movido la cifra 7.4 puntos;
- ignorar el diseño habría hecho el intervalo 2.4 veces más angosto;
- y habría convertido una caída dudosa en un titular contundente.

La estadística aplicada no es elegir la función correcta de scipy. Es saber qué le pasó
a los datos antes de llegar a tus manos.

### Apéndice opcional · Perú en Sudamérica

Si quieres el contexto regional, la API del Banco Mundial da la pobreza a paridad de
poder de compra para los 12 países. No lo ejecutamos en clase por depender de red:

```python
!pip install wbgapi -q
import wbgapi as wb
paises = ['ARG','BOL','BRA','CHL','COL','ECU','GUY','PRY','PER','SUR','URY','VEN']
# SI.POV.UMIC = pobreza a $6.85/día PPP 2017
wb.data.DataFrame('SI.POV.UMIC', paises, mrnev=1)
```

Advertencia: esas cifras usan una línea internacional distinta de la peruana, así que
**no son comparables** con el 27.6 % de hoy. Sirven para ordenar países entre sí, no para
sustituir la medición nacional.

---
*Estadística Descriptiva e Inferencial · Taller integrador · ENAHO 2023–2024 · INEI*